# Computer lab 5: robust optimization

**Mathematical Optimization, VU Amsterdam, 2026–27.**

This lab applies Lectures 9 and 10. For each robust inequality, write the
adversarial problem, derive a finite counterpart from its support function or
LP dual and check the returned plan against the original uncertainty set.

The BIM and farmer exercises draw on the MO-book companion notebooks on
[robust BIM production](https://mobook.github.io/MO-book/notebooks/08/01-bim-robust-optimization.html)
and [the farmer's problem](https://mobook.github.io/MO-book/notebooks/10/04-farmer-problem.html)
([MIT license](https://github.com/mobook/MO-book/blob/ef77f6b9768453c300d5d97e42e89fce1f5e832e/LICENSE)).


## Setup

Run the next two cells to prepare Pyomo, HiGHS and Ipopt. If prompted to
restart, rerun both cells in the new session. See the
[course setup guide](https://github.com/alessandrozocca/MO2026#computer-lab-setup)
for local setup and troubleshooting.

Use HiGHS (`lp_solver`) for the linear and mixed-integer models and Ipopt
(`nlp_solver`) for the squared norm formulations in 1(a) and 2(b). For example,
`solve_and_load(model, lp_solver)` checks the solver's termination before
loading a solution.

The inequalities $\|v\|_2\le t$ and $\|v\|_2^2\le t^2$ are equivalent when
$t\ge0$. Since Ipopt solves the squared formulation, check the original
unsquared inequality at the returned solution. Use full precision for
all feasibility checks.


In [ ]:
#@title Prepare course helpers and solvers
import hashlib
import os
from pathlib import Path
import sys
import time
import types
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

SETUP_COMMIT = '425c3f0af23e082c276df85fcde556538cc2abef'
SETUP_SHA256 = "f742478c6eaf5c5db82f4ce69b47a2adcf6f920765d88c3e7eb1b1ab72e66d28"
local_repo = os.environ.get("MO2026_REPO")
if local_repo:
    setup_location = str(Path(local_repo).expanduser() / "lab_setup.py")
    setup_source = Path(setup_location).read_bytes()
else:
    setup_location = f"https://raw.githubusercontent.com/alessandrozocca/MO2026/{SETUP_COMMIT}/lab_setup.py"
    for attempt in range(3):
        try:
            with urlopen(setup_location, timeout=30) as response:
                setup_source = response.read()
            break
        except (URLError, TimeoutError, ConnectionError) as error:
            if (isinstance(error, HTTPError) and error.code not in (408, 429, 500, 502, 503, 504)) or attempt == 2:
                raise
            time.sleep(0.5 * (attempt + 1))
    if hashlib.sha256(setup_source).hexdigest() != SETUP_SHA256:
        raise RuntimeError("Course setup checksum mismatch; the downloaded code was not executed.")

lab_setup = types.ModuleType("mo2026_lab_setup")
lab_setup.__file__ = setup_location
sys.modules[lab_setup.__name__] = lab_setup
exec(compile(setup_source, setup_location, "exec"), lab_setup.__dict__)
if local_repo and lab_setup.API_VERSION != 1:
    raise RuntimeError("This notebook requires course setup API 1.")
lab = lab_setup.setup(week=5, check=False, verbose=True)
solve_and_load = lab_setup.solve_and_load
lp_solver = lab.lp_solver
nlp_solver = lab.nlp_solver
format_number = lab_setup.format_number

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from IPython.display import display

# Use a zero relative MIP gap for the airline model.
lp_solver.options["mip_rel_gap"] = 0.0

## Exercise 1: robust BIM microchip production

Let $x_1$ and $x_2$ denote the quantities of logic and memory chips. The
nominal production problem is
$$
\begin{aligned}
    \max_x\quad &12x_1+9x_2\\
    \text{s.t.}\quad &x_1\le1000 &&\text{(silicon)},\\
    &x_2\le1500 &&\text{(germanium)},\\
    &x_1+x_2\le1750 &&\text{(plastic)},\\
    &4x_1+2x_2\le4800 &&\text{(copper)},\\
    &x\ge0.
\end{aligned}
$$

Let $z=(z_1,z_2)$ be the change from the nominal copper coefficients $(4,2)$.
The copper limit must hold for every allowed perturbation.
$$
4x_1+2x_2+z_1x_1+z_2x_2\le4800
\qquad \forall z\in Z.
$$

At $z=0$, the optimal plan from Lecture 9 is $x^*=(650,1100)$, with profit €17,700.


**(a)** Use the ball $Z=\{z\in\mathbb R^2:\|z\|_2\le\Omega\}$ with
$\Omega=0.1$. Derive the robust counterpart of the copper inequality and
solve the model in Pyomo. See Lecture 9 and Sheet 5 Exercises 2–3.


In [ ]:
# Solve the ball counterpart with Ipopt and check the unsquared copper inequality.

**(b)** Use the box $Z=[-\rho,\rho]^2$ with $\rho=0.1$. Derive the
robust counterpart using the box dual in Lecture 10 and solve the model.


In [ ]:
# Solve the box counterpart with HiGHS.

**(c)** For each returned plan and the nominal plan, maximize copper use
over the original ball and box.
Report the worst loads and the slack against 4,800. Which plans are feasible
under each set?

In [ ]:
# Check the three plans over the original sets.

**(d) Additional practice.** Lecture 9's third uncertainty set is
$$
\{z:\|z\|_\infty\le0.1,\ \|z\|_1\le0.1\}.
$$
Write $z=0.1\zeta$ with $\zeta\in Z(1)$ from Sheet 5 Exercise 6(b).
Show that the maximum increase in copper use is $0.1\max(x_1,x_2)$.
Recover the two copper inequalities from Lecture 9 and solve the LP.
Report the worst copper load and its slack against 4,800 at the
full-precision solution.


In [ ]:
# Add both copper inequalities, solve and check the worst load.


## Exercise 2: robust planning for the farmer

A farmer can plant up to 500 acres of wheat, corn and sugar beets.
Use the costs, feed requirements and prices from Lab 4 Exercise 1(a).

| Crop | Planting cost (€/acre) | Feed needed (tons) | Sale price (€/ton) | Purchase price (€/ton) |
|:---|---:|---:|---:|---:|
| Wheat | 150 | 200 | 170 | 238 |
| Corn | 230 | 240 | 150 | 210 |
| Sugar beets | 260 | 0 | 36 up to 6,000 tons, 10 above | Not purchased |

The nominal yields $\bar a_j$ are $(2.5,3,20)$ tons per acre, in crop order.
For each crop $j$, write the yield as $a_j=\bar a_j+0.2\bar a_j z_j$, with
$z_j\in[-1,1]$.

**(a)** The model is static. Choose planting, purchases and sales before
yields are known, with each balance holding for every yield vector in the
box. As in Lab 4, surplus harvest may be left unused. Derive the robust
counterparts and solve the model in Pyomo. Check the three crop balances
at the worst yields for the returned plan.


In [ ]:
# Solve the yield-box model and check the crop balances at the worst yields.

Return to the fixed yields $(2.5,3,20)$. Purchasing and selling prices
for wheat and corn now vary in two separate two-dimensional balls:

$$
p_{\rm buy}=(238,210)^\top+z_{\rm buy},\qquad
p_{\rm sell}=(170,150)^\top+z_{\rm sell},
$$
$$
\|z_{\rm buy}\|_2\le\Omega_{\rm buy}=30,\qquad
\|z_{\rm sell}\|_2\le\Omega_{\rm sell}=6.
$$

The joint uncertainty set is the Cartesian product of these balls. Beet prices
remain fixed. All planting, purchase and sale decisions are chosen before
prices are known.

**(b)** Use the epigraph step on Lecture 9 slide 37 as a guide. Here the
objective is profit, so maximize a lower bound. Derive the robust counterpart,
solve it in Pyomo and compare the acreage and profit with the nominal
fixed-price solution. At the returned plan, evaluate the two norm inequalities
of your counterpart without squaring and report their slacks.


In [ ]:
# Compare the nominal and robust price models, then check both unsquared inequalities.

**(c) Fixed trades and reoptimized trades.** At the fixed prices from (a),
use a tighter yield box of $\pm10\%$. An optimal static plan plants
$(800/9,700/9,1000/3)$ acres of wheat, corn and beets, buys 30 tons of corn,
and sells 6,000 tons of beets. There are no other trades. The plan guarantees
€91,811.11 in this box.

Apply the fixed trades in the three weather cases with yield factors $0.8$,
$1$ and $1.2$. Report the slack or shortfall in each crop balance and
explain why the trades are feasible in good weather.

Then fix the three acreages of this plan in the supplied `farmer_model()`,
as in Lab 4 Exercise 1(d), and solve once to reoptimize the trades. Report
the profit in each scenario.


In [ ]:
#@title Farmer data and model from Lab 4
CROPS = ['W', 'C', 'S']
crop_names = {'W': 'Wheat', 'C': 'Corn', 'S': 'Sugar beets'}
planting_cost = {'W': 150.0, 'C': 230.0, 'S': 260.0}
mean_yield = {'W': 2.5, 'C': 3.0, 'S': 20.0}
feed = {'W': 200.0, 'C': 240.0}
sale_price = {'W': 170.0, 'C': 150.0}
purchase_price = {'W': 238.0, 'C': 210.0}
weather = {'H': 1.2, 'M': 1.0, 'L': 0.8}


def farmer_model(yield_factors=None, price_factors=None, probabilities=None):
    factors = dict(weather if yield_factors is None else yield_factors)
    if not factors or any(not np.isfinite(v) or v < 0 for v in factors.values()):
        raise ValueError('Supply nonnegative yields for at least one scenario.')
    prices = {s: 1.0 for s in factors} if price_factors is None else dict(price_factors)
    weights = {s: 1.0 for s in factors} if probabilities is None else dict(probabilities)
    if set(prices) != set(factors) or set(weights) != set(factors):
        raise ValueError('Scenario labels must agree.')
    if any(not np.isfinite(v) or v <= 0 for v in prices.values()):
        raise ValueError('Price factors must be positive.')
    if any(not np.isfinite(v) or v < 0 for v in weights.values()) or sum(weights.values()) <= 0:
        raise ValueError('Scenario weights must be nonnegative with a positive sum.')
    total_weight = sum(weights.values())

    m = pyo.ConcreteModel('Farmer')
    m.CROPS = pyo.Set(initialize=CROPS)
    m.FEED = pyo.Set(initialize=['W', 'C'])
    m.SCENARIOS = pyo.Set(initialize=list(factors))
    m.probability = pyo.Param(m.SCENARIOS, initialize={s: weights[s] / total_weight for s in factors})
    m.plant = pyo.Var(m.CROPS, domain=pyo.NonNegativeReals, bounds=(0, 500))
    m.land = pyo.Constraint(expr=sum(m.plant[c] for c in m.CROPS) <= 500)
    m.buy = pyo.Var(m.FEED, m.SCENARIOS, domain=pyo.NonNegativeReals)
    m.sell = pyo.Var(m.CROPS, m.SCENARIOS, domain=pyo.NonNegativeReals)
    m.sell_extra = pyo.Var(m.SCENARIOS, domain=pyo.NonNegativeReals)

    # Crop balances and the beet quota.
    @m.Constraint(m.FEED, m.SCENARIOS)
    def feed_balance(m, c, s):
        return mean_yield[c] * factors[s] * m.plant[c] + m.buy[c, s] - m.sell[c, s] >= feed[c]

    @m.Constraint(m.SCENARIOS)
    def beet_balance(m, s):
        return m.sell['S', s] + m.sell_extra[s] <= mean_yield['S'] * factors[s] * m.plant['S']

    @m.Constraint(m.SCENARIOS)
    def beet_quota(m, s):
        return m.sell['S', s] <= 6000

    m.first_stage_profit = pyo.Expression(expr=-sum(planting_cost[c] * m.plant[c] for c in m.CROPS))

    @m.Expression(m.SCENARIOS)
    def recourse_profit(m, s):
        return (sum(prices[s] * (sale_price[c] * m.sell[c, s] - purchase_price[c] * m.buy[c, s])
                    for c in m.FEED)
                + 36 * m.sell['S', s] + 10 * m.sell_extra[s])

    @m.Expression(m.SCENARIOS)
    def scenario_profit(m, s):
        return m.first_stage_profit + m.recourse_profit[s]

    m.total_expected_profit = pyo.Objective(
        expr=sum(m.probability[s] * m.scenario_profit[s] for s in m.SCENARIOS),
        sense=pyo.maximize,
    )
    return m


In [ ]:
# Use mean_yield and weather to check the balances for the fixed trades.
# Fix plant[c] for each crop in farmer_model(), solve once, and read scenario_profit.


## Exercise 3: portfolio selection with budget uncertainty

A broker invests €1,000 among $n=100$ assets. Let $x_j\ge0$ be the amount
invested in asset $j$, with $\sum_j x_j=1000$. Asset $j$ has gross return
$R_j=r_j+s_j z_j$, where
$$
r_j=1.15+j(0.05/100),\qquad s_j=(0.05/300)\sqrt{45300j}.
$$
At nominal returns, asset 100 is best. With all returns at their lower
bounds, asset 1 is best. In (a) and (b), use the symmetric budget set from
Sheet 5 Exercise 6(b).
$$
Z(\Gamma)=\{z:\|z\|_\infty\le1,\ \|z\|_1\le\Gamma\},
\qquad 0\le\Gamma\le n.
$$

**(a)** Derive the robust counterpart and solve for $\Gamma=20$.
You may maximize $\tau$ subject to $R^\top x\ge\tau$ for every
$z\in Z(\Gamma)$. Terminal wealth includes the initial capital.
Net profit is terminal wealth minus €1,000. For the returned holdings,
find a worst-case perturbation in the original set and compare its terminal
wealth with the bound $\tau$.

In [ ]:
# Solve the counterpart at Gamma = 20 and recover a worst-case perturbation.

**(b)** Predict how guaranteed wealth changes as the uncertainty set grows.
Compare the portfolios for integer budgets from 0 to 50, the fractional
budget $12.5$ and the full box at $\Gamma=100$. Plot wealth and the number
of assets held.

The budget bounds $\|z\|_1$, so several partial deviations can share one
unit. At $\Gamma=12.5$, fix each holding at €10 and solve for guaranteed
wealth. Sort the coefficients $s_jx_j$ in decreasing order. Compare the
loss given by the dual with the sum of the 12 largest coefficients plus
half of the 13th.


In [ ]:
# Sweep the budgets, plot the results and check the fixed portfolio at Gamma = 12.5.

**(c)** This part uses a separate two-dimensional example. Serving demands
$d_1,d_2$ uses $d_1+2d_2$ units of capacity, with a limit of 22. Use the
one-sided budget set with $\Gamma=1$.
$$
d=(4,5)+(2\zeta_1,4\zeta_2),\qquad
0\le\zeta_1,\zeta_2\le1,\qquad \zeta_1+\zeta_2\le\Gamma.
$$

Find the maximum load. A vector in the design set *dominates* an observed
demand if it is at least as large in both coordinates. An observation is
*covered* if such a vector exists. For each of $(3,9)$ and $(6,6)$, decide
whether it is covered and give a dominating vector when one exists. Check
each observed demand against capacity. Would raising the budget to $1.5$
cover $(7,5)$ with the same coordinate bounds?


In [ ]:
# Find the maximum load, then check coverage and capacity for the observations.


## Exercise 4: airline seat allocation with polyhedral uncertainty

As in Lab 4, an airline allocates 200 economy-seat units across first,
business and economy. The three classes use 2, 1.5 and 1 seat units per
seat and earn 3, 2 and 1 revenue units per ticket, respectively. Seat
counts and planned sales are nonnegative integers. Sales cannot exceed
the seats allocated to each class, and the plan must include at least
80 economy-ticket sales.

Let $s=(s_F,s_B,s_E)$ be the sales plan. A ticket in class $i$ has an
uncertain workload score $z_i$, measured relative to the cabin-service
allowance for that class. A negative score means less work than the
allowance. The airline chooses seats and sales before it knows the scores.
Its staffing plan can absorb a total workload score of at most 280, so require
$$
s^\top z\le280\qquad \forall z\in Z,
$$
where
$$
Z=\left\{z\in\mathbb R^3:
\begin{aligned}
    1&\le z_1+z_2\le2,\\
    1.2&\le z_2+z_3\le1.8,\\
    z_1&\ge-0.9,\qquad z_2\ge-1.1,\qquad z_3\ge0
\end{aligned}
\right\}.
$$

**(a)** Write the inequalities as $Dz\le d$. With $s=(29,41,80)$, solve the
adversarial LP. Check that the maximizing $z$ satisfies both pair-sum
ranges and all three coordinate lower bounds. Give a certificate $y\ge0$ that satisfies
$D^\top y=s$ and $d^\top y=s^\top z$. Report the slack $280-s^\top z$.

**(b)** Use Lecture 10's polyhedral counterpart (slides 19–20) as a guide.
Derive the robust counterpart and solve the resulting mixed-integer linear
model in Pyomo to maximize revenue. Repeat the workload score and certificate
checks for the returned sales plan.


In [ ]:
# Check the given sales plan with the adversarial LP and a dual certificate.


In [ ]:
# Solve the robust counterpart, then repeat the checks for the returned sales.
